# MediScan AI v7.0 — JarvisLabs Quickstart
**Instance:** 8 × A6000 (384 GB VRAM total) | PyTorch template

Run each cell in order. Kernel: **MediScan v7.0 (py3.11)**

## 1. Verify Hardware

In [ ]:
import torch, platform, psutil

n_gpu = torch.cuda.device_count()
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"Python  : {platform.python_version()}")
print(f"CPUs    : {psutil.cpu_count(logical=False)} physical cores")
print(f"RAM     : {psutil.virtual_memory().total / 1e9:.0f} GB")
print(f"GPUs    : {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.0f} GB VRAM")

## 2. Show GPU Layout

In [ ]:
import sys, os
# Add project root to path
PROJECT_ROOT = os.path.abspath("../../")  # mediscan_v70_sota_production/
sys.path.insert(0, PROJECT_ROOT)

from deploy.jarvislabs.gpu_allocation import print_layout, LOAD_ORDER, SEQUENTIAL_ONLY
print_layout()
print("Load order:", LOAD_ORDER)
print("Sequential-only (big models):", sorted(SEQUENTIAL_ONLY))

## 3. Quick GPU Monitor Snapshot

In [ ]:
from deploy.jarvislabs.monitor_gpus import run
run(once=True)

## 4. Initialise MediScan Engine
This registers all 16 model wrappers but does **not** load weights yet.

In [ ]:
import os

# Point to the 8×A6000 hardware config
os.environ['MEDISCAN_MAX_RESIDENT_MODELS']         = '0'
os.environ['MEDISCAN_AUTO_UNLOAD_AFTER_INFERENCE'] = '0'
os.environ['MEDISCAN_SEQUENTIAL_HEAVY_MODELS']     = '1'
os.environ['HF_HOME']                              = '/home/hf_cache'
os.environ['PYTORCH_CUDA_ALLOC_CONF']              = 'max_split_size_mb:512,expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM']               = 'false'

from mediscan_v70.main import MediScanEngine
engine = MediScanEngine()
print(f"Registered models: {list(engine.models.keys())}")

## 5. Sequential Model Pre-load
Loads every model in the correct order:
- **Sequential**: hulu_med_32b → medgemma_27b → medix_r1_30b (multi-GPU, load one at a time)
- **Parallel**: remaining single-GPU models (loaded concurrently)

⏱ First run: ~20-40 min (downloads from HuggingFace Hub)  
⏱ Cached runs: ~3-5 min

In [ ]:
from deploy.jarvislabs.sequential_loader import run as preload

# Remove models you don't want to load from skip list
# e.g. skip=['hulu_med_32b'] to save VRAM during testing
preload(engine, dry_run=False, skip=[])

## 6. VRAM Check After Load

In [ ]:
from deploy.jarvislabs.monitor_gpus import run
run(once=True)

## 7. Run Inference Test
Analyse a sample chest X-ray image.

In [ ]:
from PIL import Image
import urllib.request, io

# Load a public sample CXR (replace with your own image path)
# img = Image.open('/home/dipankar/my_xray.jpg')

# For quick testing, create a dummy 512×512 greyscale image
import numpy as np
dummy_arr = np.random.randint(0, 200, (512, 512), dtype=np.uint8)
img = Image.fromarray(dummy_arr, mode='L').convert('RGB')
img.save('/tmp/test_cxr.png')
print('Test image saved to /tmp/test_cxr.png')

In [ ]:
import time

t0 = time.time()
result = engine.analyze(
    image_path='/tmp/test_cxr.png',
    query='Describe any abnormalities visible in this chest X-ray.',
    patient_context={'age': 45, 'sex': 'M'},
)
print(f"Inference time: {time.time()-t0:.1f}s")
print("\n--- Result ---")
import json
print(json.dumps(result, indent=2, default=str))

## 8. Start API Server (in background)
Runs on port **8080** — the custom HTTP port you added in JarvisLabs Advanced Settings.
Access it via the custom port URL shown in your JarvisLabs dashboard.

In [ ]:
# Run this cell to start the server in a background thread.
# Port 8080 = the custom HTTP port you entered in JarvisLabs Advanced Settings.
import threading, uvicorn
from mediscan_v70.api_server import app

config = uvicorn.Config(app, host='0.0.0.0', port=8080, log_level='info')
server = uvicorn.Server(config)
t = threading.Thread(target=server.run, daemon=True)
t.start()
print('Server running on http://0.0.0.0:8080')
print('Docs: http://0.0.0.0:8080/docs')

In [ ]:
# Quick health-check
import requests
r = requests.get('http://localhost:8080/health')
print(r.status_code, r.json())